## Data Cleaning & Preparation
`It is designed to solve a core financial security problem—detecting fraudulent transactions by cleaning messy data instead of analyzing garbage input`.

Many financial companies struggle with raw transaction logs full of missing values, duplicate payments, and broken formats. If left uncleaned, these errors hide subtle fraud patterns and trick security systems into flagging honest customers. This project builds a Python data pipeline to clean, fix, and organize raw transaction records into a accurate dataset. This is achieved by systematically handling missing records, dropping duplicate entries, and standardizing column types... The results become deeply reliable for businesses when their security systems run on pristine, clean data rather than noisy guesswork!


### A Referral Example:

It is believed in a running business that a missing customer location means the transaction is fraudulent! Well, this might sound plausible, but blindly deleting those rows or replacing them with random guesses can ruin your entire fraud model.

So, the Data Scientist must first inspect the data, understand *why* the values are missing, and clean them properly. Only when the dataset is sanitized can we say:

*"The data is analysis-ready."* That is exactly what we are going to do here...

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

train_path = Path("../Datasets_MLmodels/Chosen_Datasets/FraudDetection/fraudTrain.csv")
test_path = Path("../Datasets_MLmodels/Chosen_Datasets/FraudDetection/fraudTest.csv")

fraud_train = pd.read_csv(train_path)
fraud_test = pd.read_csv(test_path)

print("Datasets successfully loaded!")

Datasets successfully loaded!


In [2]:
train_clean = fraud_train.copy()
test_clean = fraud_test.copy()

In [3]:
train_clean.shape

(1296675, 23)

In [4]:
test_clean.shape

(555719, 23)

In [5]:
train_clean.columns.tolist()

['Unnamed: 0',
 'trans_date_trans_time',
 'cc_num',
 'merchant',
 'category',
 'amt',
 'first',
 'last',
 'gender',
 'street',
 'city',
 'state',
 'zip',
 'lat',
 'long',
 'city_pop',
 'job',
 'dob',
 'trans_num',
 'unix_time',
 'merch_lat',
 'merch_long',
 'is_fraud']

In [6]:
test_clean.columns.tolist()

['Unnamed: 0',
 'trans_date_trans_time',
 'cc_num',
 'merchant',
 'category',
 'amt',
 'first',
 'last',
 'gender',
 'street',
 'city',
 'state',
 'zip',
 'lat',
 'long',
 'city_pop',
 'job',
 'dob',
 'trans_num',
 'unix_time',
 'merch_lat',
 'merch_long',
 'is_fraud']

In [7]:
train_columns = set(train_clean.columns)
test_columns = set(test_clean.columns)

only_train = train_columns - test_columns
only_test = test_columns - train_columns

In [8]:
def standardize_column_names(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df
train_clean = standardize_column_names(train_clean)
test_clean = standardize_column_names(test_clean)    

In [9]:
index_columns = [
    column
    for column in train_clean.columns
    if column.startswith("unnamed:")
]
print("\nAccidental index columns detected:")
print(index_columns if index_columns else "None")

for column in index_columns:
    if column in train_clean.columns:
        train_clean.drop(columns=column, inplace=True)
    if column in test_clean.columns:
        test_clean.drop(columns=column, inplace=True)


Accidental index columns detected:
['unnamed:_0']


In [10]:
train_clean.dtypes

trans_date_trans_time     object
cc_num                     int64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
dtype: object

In [11]:
test_clean.dtypes

trans_date_trans_time     object
cc_num                     int64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
dtype: object

In [12]:
def clean_string_columns(df):
    df = df.copy()
    object_columns = df.select_dtypes(include="object").columns
    for column in object_columns:
        df[column] = df[column].str.strip()
    return df
    
train_clean = clean_string_columns(train_clean)
test_clean = clean_string_columns(test_clean)

In [13]:
datetime_columns = ["trans_date_trans_time", "dob"]

for column in datetime_columns:
    if column in train_clean.columns:
        train_clean[column] = pd.to_datetime(train_clean[column], errors="coerce")
    if column in test_clean.columns:
        test_clean[column] = pd.to_datetime(test_clean[column], errors="coerce")

In [14]:
numeric_columns = [
    "cc_num", "amt", "zip", "lat", "long", "city_pop",
    "unix_time", "merch_lat", "merch_long", "is_fraud"
]
for column in numeric_columns:
    if column in train_clean.columns:
        train_clean[column] = pd.to_numeric(train_clean[column], errors="coerce")
    if column in test_clean.columns:
        test_clean[column] = pd.to_numeric(test_clean[column], errors="coerce")

In [15]:
missing_train_before = train_clean.isna().sum()
missing_test_before = test_clean.isna().sum()

In [16]:
print("\nTraining missing values:")
print(
    missing_train_before[missing_train_before > 0]
    if (missing_train_before > 0).any()
    else "No missing values"
)

print("\nTesting missing values:")
print(
    missing_test_before[missing_test_before > 0]
    if (missing_test_before > 0).any()
    else "No missing values"
)


Training missing values:
No missing values

Testing missing values:
No missing values


In [17]:
def handle_missing_values(df):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=np.number).columns
    categorical_cols = df.select_dtypes(include="object").columns

    for column in numeric_cols:
        if df[column].isna().sum() > 0:
            median_value = df[column].median()
            if pd.notna(median_value):
                df[column] = df[column].fillna(median_value)
    datetime_cols = df.select_dtypes(
        include=["datetime64[ns]"]
    ).columns

    for column in datetime_cols:
        if df[column].isna().sum() > 0:
            mode_values = df[column].mode()
            if len(mode_values) > 0:
                df[column] = df[column].fillna(mode_values.iloc[0])

    for column in categorical_cols:
        if df[column].isna().sum() > 0:
            mode_values = df[column].mode()
            if len(mode_values) > 0:
                df[column] = df[column].fillna(mode_values.iloc[0])
    return df            

In [18]:
train_clean = handle_missing_values(train_clean)
test_clean = handle_missing_values(test_clean)

In [19]:
missing_train_after = train_clean.isna().sum()
missing_test_after = test_clean.isna().sum()

print("\nMissing values after cleaning:")

print("\nTraining:")
print(
    missing_train_after[missing_train_after > 0]
    if (missing_train_after > 0).any()
    else "No missing values"
)

print("\nTesting:")
print(
    missing_test_after[missing_test_after > 0]
    if (missing_test_after > 0).any()
    else "No missing values"
)


Missing values after cleaning:

Training:
No missing values

Testing:
No missing values


In [20]:
duplicate_train_before = train_clean.duplicated().sum()
duplicate_test_before = test_clean.duplicated().sum()

print("\nDuplicate rows in training:", duplicate_train_before)
print("Duplicate rows in testing :", duplicate_test_before)


Duplicate rows in training: 0
Duplicate rows in testing : 0


In [21]:
train_clean = train_clean.drop_duplicates().reset_index(drop=True)
test_clean = test_clean.drop_duplicates().reset_index(drop=True)

In [22]:
duplicate_train_after = train_clean.duplicated().sum()
duplicate_test_after = test_clean.duplicated().sum()

print("\nDuplicates after cleaning:")
print("Training:", duplicate_train_after)
print("Testing :", duplicate_test_after)


Duplicates after cleaning:
Training: 0
Testing : 0


In [23]:
print("\n" + "=" * 70)
print("TARGET VARIABLE VALIDATION")
print("=" * 70)

if "is_fraud" not in train_clean.columns:
    raise KeyError("Expected target column 'is_fraud' was not found.")

if "is_fraud" not in test_clean.columns:
    raise KeyError("Expected target column 'is_fraud' was not found.")

print("\nTraining target values:")
print(train_clean["is_fraud"].value_counts(dropna=False))

print("\nTesting target values:")
print(test_clean["is_fraud"].value_counts(dropna=False))


TARGET VARIABLE VALIDATION

Training target values:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Testing target values:
is_fraud
0    553574
1      2145
Name: count, dtype: int64


In [24]:
valid_target_values = {0, 1}

invalid_train_target = set(
    train_clean["is_fraud"].dropna().unique()
) - valid_target_values

invalid_test_target = set(
    test_clean["is_fraud"].dropna().unique()
) - valid_target_values

print("\nInvalid target values in training:")
print(invalid_train_target if invalid_train_target else "None")

print("\nInvalid target values in testing:")
print(invalid_test_target if invalid_test_target else "None")


Invalid target values in training:
None

Invalid target values in testing:
None


In [25]:
print("\n" + "=" * 70)
print("TRANSACTION AMOUNT VALIDATION")
print("=" * 70)

if "amt" in train_clean.columns:

    negative_train_amounts = (train_clean["amt"] < 0).sum()
    negative_test_amounts = (test_clean["amt"] < 0).sum()

    print("Negative transaction amounts in training:", negative_train_amounts)
    print("Negative transaction amounts in testing :", negative_test_amounts)


TRANSACTION AMOUNT VALIDATION
Negative transaction amounts in training: 0
Negative transaction amounts in testing : 0


In [26]:
if "trans_num" in train_clean.columns:
    duplicate_trans_num_train = train_clean["trans_num"].duplicated().sum()
    print("Repeated transaction IDs in training:", duplicate_trans_num_train)

if "trans_num" in test_clean.columns:
    duplicate_trans_num_test = test_clean["trans_num"].duplicated().sum()
    print("Repeated transaction IDs in testing:", duplicate_trans_num_test)

Repeated transaction IDs in training: 0
Repeated transaction IDs in testing: 0


In [27]:
if "trans_num" in train_clean.columns and "trans_num" in test_clean.columns:
    train_transaction_ids = set(train_clean["trans_num"].dropna())
    test_transaction_ids = set(test_clean["trans_num"].dropna())
    overlapping_transactions = (train_transaction_ids & test_transaction_ids)

    print("Transactions appearing in both datasets:", len(overlapping_transactions))
else:
    print("trans_num not available; transaction overlap could not be checked using transaction ID.")


Transactions appearing in both datasets: 0


In [28]:
print("\n" + "=" * 70)
print("DATA TYPES AFTER CLEANING")
print("=" * 70)

print("\nTraining:")
print(train_clean.dtypes)

print("\nTesting:")
print(test_clean.dtypes)


DATA TYPES AFTER CLEANING

Training:
trans_date_trans_time    datetime64[ns]
cc_num                            int64
merchant                         object
category                         object
amt                             float64
first                            object
last                             object
gender                           object
street                           object
city                             object
state                            object
zip                               int64
lat                             float64
long                            float64
city_pop                          int64
job                              object
dob                      datetime64[ns]
trans_num                        object
unix_time                         int64
merch_lat                       float64
merch_long                      float64
is_fraud                          int64
dtype: object

Testing:
trans_date_trans_time    datetime64[ns]
cc_num            

In [29]:
print("\nTraining shape:")
print(train_clean.shape)

print("\nTesting shape:")
print(test_clean.shape)



Training shape:
(1296675, 22)

Testing shape:
(555719, 22)


In [30]:
final_missing_train = train_clean.isna().sum().sum()
final_missing_test = test_clean.isna().sum().sum()

print("\nTotal missing values remaining:")
print("Training:", final_missing_train)
print("Testing :", final_missing_test)


Total missing values remaining:
Training: 0
Testing : 0


In [31]:
final_duplicates_train = train_clean.duplicated().sum()
final_duplicates_test = test_clean.duplicated().sum()

print("\nTotal duplicate rows remaining:")
print("Training:", final_duplicates_train)
print("Testing :", final_duplicates_test)


Total duplicate rows remaining:
Training: 0
Testing : 0


In [32]:
final_train_target_values = set(
    train_clean["is_fraud"].dropna().unique()
)

final_test_target_values = set(
    test_clean["is_fraud"].dropna().unique()
)

print("\nFinal target values:")
print("Training:", final_train_target_values)
print("Testing :", final_test_target_values)


Final target values:
Training: {np.int64(0), np.int64(1)}
Testing : {np.int64(0), np.int64(1)}


In [33]:
quality_report = pd.DataFrame({

    "metric": [
        "Training rows before cleaning", "Training rows after cleaning",
        "Testing rows before cleaning", "Testing rows after cleaning",
        "Training duplicate rows removed", "Testing duplicate rows removed",
        "Training missing values remaining", "Testing missing values remaining",
        "Training invalid target values", "Testing invalid target values",
        "Training negative transaction amounts",  "Testing negative transaction amounts",
        "Overlapping transaction IDs"
    ],

    "value": [

        len(fraud_train), len(train_clean), len(fraud_test), len(test_clean),
         duplicate_train_before - duplicate_train_after, 
         duplicate_test_before - duplicate_test_after, final_missing_train,
        final_missing_test, len(invalid_train_target), len(invalid_test_target),

        (
            (train_clean["amt"] < 0).sum()
            if "amt" in train_clean.columns
            else np.nan
        ),

        (
            (test_clean["amt"] < 0).sum()
            if "amt" in test_clean.columns
            else np.nan
        ),

        (
            len(overlapping_transactions)
            if "trans_num" in train_clean.columns
            and "trans_num" in test_clean.columns
            else np.nan
        )
    ]
})

In [34]:
output_dir = Path("../Datasets_MLmodels/Fraud/P03")
output_dir.mkdir(parents=True, exist_ok=True)

train_output_path = output_dir / "fraudTrain_clean.csv"
test_output_path = output_dir / "fraudTest_clean.csv"
report_output_path = output_dir / "cleaning_report.csv"

train_clean.to_csv(train_output_path, index=False)
test_clean.to_csv(test_output_path, index=False)
quality_report.to_csv(report_output_path, index=False)    

print(f"Files successfully exported to: {output_dir}")

Files successfully exported to: outputs


In [34]:
print("\n" + "=" * 70)
print("\t\t\t DATA QUALITY REPORT")
print("=" * 70)
print(quality_report.to_string(index=False))


			 DATA QUALITY REPORT
                               metric   value
        Training rows before cleaning 1296675
         Training rows after cleaning 1296675
         Testing rows before cleaning  555719
          Testing rows after cleaning  555719
      Training duplicate rows removed       0
       Testing duplicate rows removed       0
    Training missing values remaining       0
     Testing missing values remaining       0
       Training invalid target values       0
        Testing invalid target values       0
Training negative transaction amounts       0
 Testing negative transaction amounts       0
          Overlapping transaction IDs       0
